In [ ]:
# @title Github Data Loader, Tokenizer Training, and Dataset Builder

import os

try:
    import google.colab
    REPO_URL = "https://github.com/wtheisen/nd-cse-10124-lectures.git"

    REPO_NAME = "/content/nd-cse-10124-lectures"
    L_PATH = "nd-cse-10124-lectures"

    %cd /content/
    !rm -r {REPO_NAME}

    # Clone repo
    if not os.path.exists(REPO_NAME):
        !git clone {REPO_URL}

        # cd into the data folder
        %cd {L_PATH}
        !pwd

except ImportError:
    print("Unable to download repo, either:")
    print("\tA.) You're not on colab")
    print("\tB.) It has already been cloned")

!pwd

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

import irishGPT as iGPT
r_t = iGPT.tokenizer.Regex_Tokenizer()
r_t.load('Datasets/shakespeare_vocab.json')

dataset = iGPT.dataset.IrishChatDataset('Datasets/shakespeare.txt', r_t)

/content
rm: cannot remove '/content/nd-cse-10124-lectures': No such file or directory
Cloning into 'nd-cse-10124-lectures'...
remote: Enumerating objects: 396, done.
remote: Counting objects: 100% (129/129), done.
remote: Compressing objects: 100% (84/84), done.
remote: Total 396 (delta 75), reused 96 (delta 45), pack-reused 267 (from 1)
Receiving objects: 100% (396/396), 34.32 MiB | 16.48 MiB/s, done.
Resolving deltas: 100% (251/251), done.
/content/nd-cse-10124-lectures
/content/nd-cse-10124-lectures
/content/nd-cse-10124-lectures
device: cuda


In [ ]:
# @title Generative Recurrent Neural Network

import torch
import torch.nn as nn
import torch.nn.functional as F

class SLM(nn.Module):
    def __init__(self, device='cpu'):
        super().__init__()
        self.device = device

        self.embedding = nn.Embedding(512, 128)
        self.rnn = nn.RNN(128, 64, batch_first=True)
        self.output = nn.Linear(64, 512)

    def forward(self, x, h0=None):
        x = self.embedding(x)  # (B,T,E)

        if h0 is None:
            out, h = self.rnn(x)       # out: (B,T,H)
        else:
            out, h = self.rnn(x, h0)   # use carried hidden state

        logits = self.output(out)      # (B,T,V)
        return logits, h

    @torch.no_grad()
    def _sample_logits(self, logits_last, temperature=0.8, top_k=512):
        if logits_last.dim() == 2:
            logits_last = logits_last[0]  # (V,)

        logits_last = logits_last / temperature

        vals, idx = torch.topk(logits_last, top_k)
        filtered = torch.full_like(logits_last, float("-inf"))
        filtered[idx] = vals
        logits_last = filtered

        probs = F.softmax(logits_last, dim=-1)              # (V,)
        next_id = torch.multinomial(probs, num_samples=1)   # (1,)
        return int(next_id.item())

    @torch.no_grad()
    def generate(self, prompt_tokens, max_new_tokens=80, top_k=512, temperature=0.8):
        self.eval()
        x = prompt_tokens
        logits, h = self.forward(x)

        for _ in range(max_new_tokens):
            next_logits = logits[:, -1, :]        # (1,V)
            next_id = self._sample_logits(next_logits, temperature, top_k)

            x_new = torch.tensor([[next_id]], dtype=torch.long, device=self.device)  # (1,1)
            logits, h = self.forward(x_new, h0=h)        # logits: (1,1,V)

            x = torch.cat([x, x_new], dim=1)

            if x_new == 258:
                break

        return x[0].tolist()

In [ ]:
# @title SLM Training Loop and Helper Function

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

def train_slm(dataset, epochs=25, batch_size=64, lr=3e-4, grad_clip=1.0):
    device = dataset.device
    V = len(dataset.tokenizer.vocab)

    model = SLM(device=device).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=lr)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=dataset.collate,
    )

    for epoch in range(1, epochs + 1):
        model.train()
        total_loss_sum = 0.0   # sum over tokens
        total_tokens = 0

        for X, Y_onehot, mask in loader:
            opt.zero_grad(set_to_none=True)

            logits, _ = model(X)                    # (B,T,V)
            log_probs = F.log_softmax(logits, dim=-1)

            # per-position CE: -(y · log p)
            per_pos_loss = -(Y_onehot * log_probs).sum(dim=-1)   # (B,T)

            # ignore padding positions
            per_pos_loss = per_pos_loss * mask.float()           # (B,T)

            loss_sum = per_pos_loss.sum()                        # scalar
            n_tokens = mask.sum().item()

            if n_tokens == 0:
                continue

            loss = loss_sum / n_tokens                           # average per real token
            loss.backward()

            if grad_clip is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            opt.step()

            total_loss_sum += loss_sum.item()
            total_tokens += n_tokens

        avg_loss = total_loss_sum / max(1, total_tokens)
        ppl = float(torch.exp(torch.tensor(avg_loss)))
        print(f"epoch {epoch:02d} | loss/token={avg_loss:.4f} | ppl={ppl:.2f}")
        ids = dataset.tokenizer.encode("<|sos|>hello there <|eos|>")[:-1]
        prompt_tokens = torch.tensor([ids], dtype=torch.long, device=device)  # (1,T)
        tokens = model.generate(prompt_tokens, temperature=1.0)
        print('Sample Generation for epoch:')
        print('Tokens:', tokens)
        print('Text:', r_t.decode(tokens), '\n')
        model.train()

    return model

In [ ]:
# @title SLM Training

slm = train_slm(dataset)

epoch 01 | loss/token=4.3320 | ppl=76.09
Sample Generation for epoch:
Tokens: [257, 260, 276, 111, 270, 267, 32, 323, 104, 121, 46, 258]
Text: <|sos|>hello there imhy.<|eos|> 

epoch 02 | loss/token=3.6095 | ppl=36.95
Sample Generation for epoch:
Tokens: [257, 260, 276, 111, 270, 267, 32, 300, 325, 313, 299, 97, 276, 44, 32, 275, 116, 111, 270, 293, 322, 278, 379, 410, 114, 262, 500, 32, 114, 308, 351, 44, 258]
Text: <|sos|>hello there anch my heall, orto the piron as Grounter rather,<|eos|> 

epoch 03 | loss/token=3.4418 | ppl=31.24
Sample Generation for epoch:
Tokens: [257, 260, 276, 111, 270, 267, 32, 74, 101, 345, 121, 297, 452, 285, 116, 115, 111, 270, 263, 111, 269, 266, 347, 433, 108, 303, 393, 282, 273, 116, 486, 116, 387, 330, 116, 309, 116, 44, 292, 327, 342, 363, 44, 263, 317, 110, 274, 297, 102, 483, 46, 382, 121, 284, 317, 261, 108, 340, 110, 63, 258]
Text: <|sos|>hello there Jeady of our otso the sond was splow will dist contain'st bet, you not with him, sayner offres. By

In [22]:
# @title SLM Prompting

prompt="<|sos|>Thou<|eos|>"

ids = r_t.encode(prompt)[:-1]
prompt_tokens = torch.tensor([ids], dtype=torch.long, device=device)  # (1,T)

tokens = slm.generate(prompt_tokens)

print(r_t.decode(tokens))

<|sos|>Thou'll have my faulted but for away<|eos|>
